# 🔄 Kuramoto Phase Oscillator Network

The Kuramoto model describes synchronization of coupled oscillators:

$$\dot{\theta}_i = \omega_i + \sum_j W_{ij} \sin(\theta_j - \theta_i)$$

The **order parameter** quantifies synchronization:

$$r e^{i\psi} = \frac{1}{N}\sum_j e^{i\theta_j}$$

$r=1$ = perfect synchrony, $r=0$ = complete asynchrony.

This implementation uses sparse CSR matrix for connectivity and SIMD for phase updates.


## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import taas_checkmate as ne
except ImportError:
    import subprocess
    subprocess.run(["maturin", "develop", "--release", "-q"], check=True)
    import taas_checkmate as ne

plt.style.use("dark_background")
N = 256
N_PAD = (N + 7) // 8 * 8
DT = 0.5

rng = np.random.default_rng(42)
print("✅ Setup complete")


## Network Setup: Sparse Connectivity

In [ ]:
# Random sparse connectivity
SPARSITY = 0.08
edges = []
for i in range(N_PAD):
    for j in range(N_PAD):
        if i != j and rng.random() < SPARSITY:
            edges.append((i, j, rng.uniform(0.1, 1.0)))

edges.sort(key=lambda e: e[0])
row_ptr = [0]
col = []
weights = []
count = 0
for i in range(N_PAD):
    while count < len(edges) and edges[count][0] == i:
        col.append(edges[count][1])
        weights.append(edges[count][2])
        count += 1
    row_ptr.append(len(col))

row_ptr = np.array(row_ptr, dtype=np.int32)
col = np.array(col, dtype=np.int32)
weights = np.array(weights, dtype=np.float32)

# Phase initialization: two clusters
phases = np.zeros(N_PAD, dtype=np.float32)
phases[:N//2] = rng.uniform(0, 0.5, N//2)  # cluster 1
phases[N//2:] = rng.uniform(np.pi - 0.5, np.pi + 0.5, N - N//2)  # cluster 2
omegas = rng.uniform(0.5, 1.5, N_PAD).astype(np.float32)

print(f"Network: {N_PAD} oscillators, {len(col)} synapses ({SPARSITY*100:.0f}% sparse)")
print(f"Two clusters initialized: 0 rad (cluster 1) vs π rad (cluster 2)")


## Run Simulation

In [ ]:
STEPS = 2000
sin_buf = np.zeros(N_PAD, dtype=np.float32)
cos_buf = np.zeros(N_PAD, dtype=np.float32)
coup_buf = np.zeros(N_PAD, dtype=np.float32)

phase_history = np.zeros((STEPS, N), dtype=np.float32)
order_r = np.zeros(STEPS, dtype=np.float32)

from rich.progress import Progress

with Progress() as pbar:
    task = pbar.add_task("[cyan]Simulating...", total=STEPS)
    for t in range(STEPS):
        ne.kuramoto_step_sparse(phases, omegas, row_ptr, col, weights,
                                sin_buf, cos_buf, coup_buf, DT)
        phase_history[t] = phases[:N]
        # Order parameter
        r = np.abs(np.mean(np.exp(1j * phases[:N])))
        order_r[t] = r
        pbar.update(task, advance=1)

print(f"\nFinal order parameter: r = {order_r[-1]:.4f}")
print(f"  (1.0 = perfect sync, 0.0 = async)")


## Synchronization Dynamics

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

t = np.arange(STEPS) * DT

# Order parameter over time
ax1.plot(t, order_r, color="#00ff88", linewidth=1)
ax1.axhline(order_r.mean(), color="#ff6600", linestyle="--", alpha=0.5,
            label=f"Mean r = {order_r.mean():.3f}")
ax1.set_xlabel("Time (ms)")
ax1.set_ylabel("Order Parameter r")
ax1.set_title("Kuramoto Synchronization")
ax1.legend()
ax1.grid(alpha=0.1)
ax1.set_ylim(0, 1)

# Phase evolution (sample)
sample_idx = [0, 10, 50, 100, 200]
for idx in sample_idx:
    if idx < N:
        ax2.plot(t, phase_history[:, idx], linewidth=0.8,
                 label=f"θ_{idx}")
ax2.set_xlabel("Time (ms)")
ax2.set_ylabel("Phase (rad)")
ax2.set_title("Individual Phase Evolution")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.1)

plt.tight_layout()
plt.show()


## Phase Distribution (Histogram)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
snapshots = [0, 100, 500, 1000, 1500, 1999]

for ax, snap in zip(axes.flat, snapshots):
    ph = phase_history[snap]
    hist, bins = np.histogram(ph, bins=36, range=(0, 2*np.pi))
    theta = (bins[:-1] + bins[1:]) / 2
    ax.bar(theta, hist, width=2*np.pi/36, color="#00ff88",
           alpha=0.7, edgecolor="none")
    ax.set_title(f"t = {snap * DT:.0f} ms")
    ax.set_xlim(0, 2 * np.pi)
    ax.set_ylim(0, N // 4)
    ax.set_xlabel("Phase")
    ax.set_ylabel("Count")
    ax.grid(alpha=0.1)

fig.suptitle("Phase Distribution Over Time", fontsize=14)
plt.tight_layout()
plt.show()


## Phase Coherence Map

In [ ]:
# Compute coherence between areas
N_AREAS = 4
area_size = N // N_AREAS
coherence = np.zeros((N_AREAS, N_AREAS), dtype=np.float32)

for a1 in range(N_AREAS):
    for a2 in range(N_AREAS):
        p1 = phase_history[:, a1*area_size:(a1+1)*area_size]
        p2 = phase_history[:, a2*area_size:(a2+1)*area_size]
        # Mean vector of phase differences
        diff = p1[:, :, None] - p2[:, None, :]
        r = np.abs(np.mean(np.exp(1j * diff)))
        coherence[a1, a2] = r

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(coherence, cmap="plasma", vmin=0, vmax=1)
ax.set_xticks(range(N_AREAS))
ax.set_yticks(range(N_AREAS))
ax.set_xlabel("Area")
ax.set_ylabel("Area")
ax.set_title("Phase Coherence Matrix")
plt.colorbar(im, ax=ax, label="Coherence", shrink=0.8)

for i in range(N_AREAS):
    for j in range(N_AREAS):
        ax.text(j, i, f"{coherence[i,j]:.2f}",
                ha="center", va="center", fontsize=10,
                color="white" if coherence[i,j] < 0.7 else "black")

plt.tight_layout()
plt.show()
